# Phase 1: Create the miniature fraud dataset

This notebook creates a synthetic dataset that matches the project schema:
- user_id
- signup_time
- purchase_time
- purchase_value
- device_id
- source
- browser
- sex
- age
- ip_address
- class

The target variable is:
- 0 = legitimate transaction
- 1 = fraudulent transaction

We will create a realistic but small dataset for later EDA, feature engineering, and modeling.

In [2]:
import pandas as pd
import numpy as np

In [3]:
# -----------------------------
# 1) Set random seed
# -----------------------------
rng = np.random.default_rng(42)

# -----------------------------
# 2) Create synthetic data
# -----------------------------
n_rows = 1000

# User / device IDs
user_id = [f"U{idx:05d}" for idx in range(1, n_rows + 1)]
device_id = [f"D{idx:05d}" for idx in range(1, n_rows + 1)]

# Signup and purchase timestamps
base_date = pd.Timestamp("2023-01-01")
signup_time = pd.to_datetime(
    base_date + pd.to_timedelta(rng.integers(0, 365 * 2, size=n_rows), unit="D")
)

purchase_time = signup_time + pd.to_timedelta(
    rng.integers(1, 90, size=n_rows), unit="D"
)

# Purchase value in USD
purchase_value = np.round(
    rng.lognormal(mean=3.4, sigma=0.9, size=n_rows),
    2
)

# Browser and source
sources = ["SEO", "Ads", "Direct", "Referral", "Social"]
browsers = ["Chrome", "Safari", "Firefox", "Edge"]

source = rng.choice(sources, size=n_rows, p=[0.25, 0.35, 0.20, 0.10, 0.10])
browser = rng.choice(browsers, size=n_rows, p=[0.45, 0.25, 0.15, 0.15])

# Sex and age
sex = rng.choice(["M", "F"], size=n_rows, p=[0.52, 0.48])
age = rng.integers(18, 72, size=n_rows)

# IP address
ip_int = rng.integers(0, 2**32 - 1, size=n_rows)
ip_address = [
    ".".join(str((ip >> shift) & 255) for shift in (24, 16, 8, 0))
    for ip in ip_int
]

In [4]:
# -----------------------------
# 3) Create fraud probability
# -----------------------------
# Fraud is more likely when:
# - purchase value is high
# - source is Ads or Social
# - age is low or medium
# - browser/behavior combinations look suspicious
fraud_score = (
    0.8 * np.log1p(purchase_value)
    + 1.2 * (source == "Ads").astype(int)
    + 0.9 * (source == "Social").astype(int)
    + 0.7 * (browser == "Safari").astype(int)
    + 0.5 * (age < 25).astype(int)
    + 0.8 * (purchase_value > 500).astype(int)
)

fraud_prob = 1 / (1 + np.exp(-fraud_score))
fraud_prob = np.clip(fraud_prob / 4.5, 0.01, 0.35)

# Create the target with class imbalance
class_label = rng.binomial(1, fraud_prob, size=n_rows)


In [6]:
# -----------------------------
# 4) Build the DataFrame
# -----------------------------
fraud_df = pd.DataFrame({
    "user_id": user_id,
    "signup_time": signup_time,
    "purchase_time": purchase_time,
    "purchase_value": purchase_value,
    "device_id": device_id,
    "source": source,
    "browser": browser,
    "sex": sex,
    "age": age,
    "ip_address": ip_address,
    "class": class_label
})

In [12]:
# 5) Check on what we have done.
print(fraud_df.head(5))
print()
print(fraud_df["class"].value_counts())
print()
print(fraud_df["class"].value_counts(normalize=True))
print()
print("SHape: " , fraud_df.shape)

  user_id signup_time purchase_time  purchase_value device_id source  browser  \
0  U00001  2023-03-07    2023-05-18           15.36    D00001    SEO  Firefox   
1  U00002  2024-07-18    2024-09-22           68.85    D00002    Ads   Safari   
2  U00003  2024-04-22    2024-06-11           30.91    D00003    Ads   Chrome   
3  U00004  2023-11-17    2024-02-04           23.23    D00004    Ads  Firefox   
4  U00005  2023-11-13    2023-11-25           27.23    D00005    Ads     Edge   

  sex  age       ip_address  class  
0   F   22      22.99.162.8      0  
1   M   29  119.212.113.140      0  
2   F   70   187.78.144.228      0  
3   F   57   83.189.160.248      0  
4   M   58      91.93.44.92      0  

class
0    801
1    199
Name: count, dtype: int64

class
0    0.801
1    0.199
Name: proportion, dtype: float64

SHape:  (1000, 11)


In [14]:
fraud_df.to_csv("../data/processed/mini_transaction.csv", index=False)